# BERT master trajectories

Builds the **master** token table for BERT pre-training: one row per token, all patients, all channels.
D1, D2 and D3 are later obtained from it by dropping rows (see the last section).

**Inputs** (paths in `trajectory_columns.yaml`)
- `events_thinned_4h.parquet`: thinned events, with `quantile_bin`, `label`, `ontology_version`, `event_row`
- `vocabulary/vocab.parquet`: token ids (special, quantile bins, gender, units, concepts)
- MIMIC `patients.csv.gz`: gender

**Outputs** (in `trajectory_out/bert/master/`)
- `split.parquet`: train / val / test by patient (60 / 20 / 20, seed 42)
- `master_bXX.parquet`: the master table, one file per bucket of patients
- `patient_index.parquet`: subject_id, split, bucket, n_tokens

**Rules decided**
- Sequence = gender token, then the events in time order. **No `[SEP]`** anywhere.
- One event gives up to 4 tokens, always in this order: concept, quantile, `[VALUE]`, unit.
  - quantile and `[VALUE]`: only for numeric events
  - unit: only for numeric events that have a unit
- `piece` = event number per patient (0 = gender, then 1, 2, ...); all tokens of one event share it.
- `type` = gender / code / quantile / value / unit; used by the masking, not given to the model.
- Order: `age_days`, then `seconds_in_day`; **date-only events go last in their day** (null seconds);
  `event_row` breaks ties, so the order is the same on every run.
- Events with no date are removed. Patients with no events are removed.

## 1. Setup

Reads the paths, the split and the column lists from `trajectory_columns.yaml`.
`KEY` is how the vocabulary identifies a concept: the same code can exist in ICD-9 and ICD-10,
and death has no code, so the code alone is not enough.

In [ ]:
import gc
from pathlib import Path

import numpy as np
import polars as pl
import yaml

with open("trajectory_columns.yaml", "r") as f:
    cfg = yaml.load(f, Loader=yaml.SafeLoader)

OUT = Path(cfg["paths"]["out"])
EVENTS = OUT / cfg["paths"]["events"]
VOCAB = OUT / cfg["paths"]["vocab"]
MASTER = OUT / cfg["paths"]["master_dir"]
PATIENTS = Path(cfg["paths"]["patients"])
N_BUCKETS = cfg["n_buckets"]

KEY = ["source", "ontology_version", "label", "code"]  # how the vocabulary identifies a concept

MASTER.mkdir(parents=True, exist_ok=True)

## 2. Vocabulary size: choose the cut-off

A concept is kept if it is seen in at least `min_patients` patients (set in `trajectory_columns.yaml`);
rarer concepts become `[UNK]` **for the model**. Special tokens, quantile bins, gender, death and units are always kept.

The master table always stores the **full** vocabulary ids (`token_id`), so changing the cut-off never needs a rebuild.
The cut-off only produces a remap table `token_id → model_id`, applied when the data is given to the model (section 10).

$$\text{model\_id} = \begin{cases} \text{new number of the token} & \text{if kept} \\ \text{id of [UNK]} & \text{otherwise} \end{cases}$$

Kept tokens are renumbered 0, 1, 2, … without gaps, in the order of the full vocabulary,
so the special tokens keep the same ids (`[UNK]` = 1 in both).

In [ ]:
MIN_PATIENTS = cfg["min_patients"]

vocab = pl.read_parquet(VOCAB).with_columns(pl.col("token_id").cast(pl.Int64))
always = pl.col("source").is_in(["special", "quantile", "patients", "unit"])  # patients = gender and death

# vocabulary size for a few cut-offs, to help the choice
thresholds = [1, 5, 10, 20, 50, 100, 500, 1000]
n_always = vocab.filter(always).height
print(pl.DataFrame({
    "min_patients": thresholds,
    "vocab_size": [n_always + vocab.filter(~always & (pl.col("n_patients") >= t)).height for t in thresholds],
}))

# remap for the chosen cut-off: kept tokens renumbered without gaps, the others -> [UNK]
unk_full_id = vocab.filter((pl.col("source") == "special") & (pl.col("label") == "[UNK]"))["token_id"].item()
remap = (vocab
         .select("token_id", (always | (pl.col("n_patients") >= MIN_PATIENTS)).alias("kept"))
         .with_columns(model_id=pl.col("kept").cast(pl.Int64).cum_sum() - 1))
unk_model_id = remap.filter(pl.col("token_id") == unk_full_id)["model_id"].item()
remap = remap.with_columns(model_id=pl.when(pl.col("kept")).then(pl.col("model_id")).otherwise(unk_model_id))
remap.write_parquet(MASTER / f"vocab_remap_min{MIN_PATIENTS}.parquet")

print(f"min_patients = {MIN_PATIENTS}: model vocabulary size = {remap['kept'].sum():,} "
      f"(full vocabulary: {vocab.height:,})")

## 3. Vocabulary lookups

Each kind of token is looked up in its own way:

| token | looked up by | example |
|---|---|---|
| concept (events, gender, death) | `KEY` = source, ontology_version, label, code | labevents, –, creatinine, 50912 |
| quantile bin | its label | Q8 |
| unit | the unit as written in the data, spaces stripped | mg/dL |
| `[VALUE]` | fixed special id | 5 |

These are **full** vocabulary ids (no cut-off). A concept that is not found at all gets `[UNK]`;
the checks in section 9 count them (expected: 0).

In [ ]:
# concepts: events, gender and death (death has no code, so its code is null)
concept_ids = (vocab
               .filter(~pl.col("source").is_in(["special", "quantile", "unit"]))
               .select(*KEY, pl.col("token_id").alias("concept_id")))

# quantile bins Q1..Q10 and "unknown"
quantile_ids = (vocab
                .filter(pl.col("source") == "quantile")
                .select(pl.col("label").alias("quantile_bin"), pl.col("token_id").alias("quantile_id")))

# units, as written in the data
unit_ids = (vocab
            .filter(pl.col("source") == "unit")
            .select(pl.col("code").alias("unit_key"), pl.col("token_id").alias("unit_id")))

# special tokens: [VALUE] is the placeholder of the raw number, [UNK] for anything not found
special = dict(vocab.filter(pl.col("source") == "special").select("label", "token_id").iter_rows())
VALUE_ID, UNK_ID = special["[VALUE]"], special["[UNK]"]

print(f"concepts: {concept_ids.height:,}  quantile bins: {quantile_ids.height}  "
      f"units: {unit_ids.height}  [VALUE] = {VALUE_ID}  [UNK] = {UNK_ID}")

## 4. Patients and split

The split is made on **all** MIMIC patients with a fixed seed, so a patient keeps the same split
even if the events are rebuilt later. Patients without events simply never appear in the master table.

In [ ]:
def make_split(ids, fractions, seed):
    # Random split by patient: sort first, then shuffle with a fixed seed (same result on every run)
    ids = np.sort(np.asarray(ids, dtype=np.int64))
    np.random.default_rng(seed).shuffle(ids)
    n = len(ids)
    n_train, n_val = round(fractions["train"] * n), round(fractions["val"] * n)
    labels = ["train"] * n_train + ["val"] * n_val + ["test"] * (n - n_train - n_val)
    return pl.DataFrame({"subject_id": ids, "split": labels})


patients = pl.read_csv(PATIENTS, columns=["subject_id", "gender"])
split = make_split(patients["subject_id"].unique(), cfg["split"]["fractions"], cfg["split"]["seed"])
split.write_parquet(MASTER / "split.parquet")
split.group_by("split").len().sort("split")

## 5. Transformations

Small functions, each doing one step, chained with `.pipe()` in section 6.

- `bucket_of`: the bucket of a patient (same patient → same bucket, in the events and in the patients table)
- `add_seconds_in_day`: `"HH:MM:SS"` → seconds since midnight; null for date-only events
- `add_token_ids`: concept, quantile and unit ids of each event
- `order_events`: time order per patient, date-only events last in their day
- `add_piece`: event number per patient, from 1 (0 is the gender)
- `explode_event`: one event row → its tokens (concept, quantile, `[VALUE]`, unit)
- `gender_rows`: the static part, one gender token per patient

In [ ]:
def bucket_of(col="subject_id"):
    return pl.col(col).hash(seed=3) % N_BUCKETS


def add_seconds_in_day(df, col="time_of_day", out="seconds_in_day"):
    t = pl.col(col).str.to_time("%H:%M:%S")
    return df.with_columns(
        (t.dt.hour().cast(pl.Int32) * 3600
         + t.dt.minute().cast(pl.Int32) * 60
         + t.dt.second().cast(pl.Int32)).alias(out)
    ).drop(col)


def add_token_ids(df):
    return (df
            # nulls_equal: a null code (death) or a null ontology_version must match the vocabulary row
            # (older Polars versions call this argument join_nulls)
            .join(concept_ids, on=KEY, how="left", nulls_equal=True)
            .join(quantile_ids, on="quantile_bin", how="left")
            .with_columns(unit_key=pl.col("unit").str.strip_chars())
            .join(unit_ids, on="unit_key", how="left")
            .with_columns(pl.col("concept_id").fill_null(UNK_ID))
            .drop("unit_key"))


def order_events(df):
    # nulls_last: date-only events (null seconds) go after the timed events of the same day
    return df.sort(["subject_id", "age_days", "seconds_in_day", "event_row"], nulls_last=True)


def add_piece(df):
    # needs the rows already in time order
    return df.with_columns(piece=pl.int_range(1, pl.len() + 1, dtype=pl.Int64).over("subject_id"))

In [ ]:
def explode_event(df):
    has_q = pl.col("quantile_id").is_not_null()                 # numeric event
    has_unit = has_q & pl.col("unit_id").is_not_null()          # numeric event with a unit
    return (df
            .with_columns(
                token_id=pl.concat_list("concept_id", "quantile_id", pl.lit(VALUE_ID, pl.Int64), "unit_id"),
                type=pl.concat_list(pl.lit("code"), pl.lit("quantile"), pl.lit("value"), pl.lit("unit")),
                keep=pl.concat_list(pl.lit(True), has_q, has_q, has_unit),
            )
            .explode("token_id", "type", "keep")
            .filter(pl.col("keep"))
            # the raw number stays only on the [VALUE] row
            .with_columns(value=pl.when(pl.col("type") == "value")
                                  .then(pl.col("value").cast(pl.Float64, strict=False)))
            .drop("keep", "concept_id", "quantile_id", "unit_id", "quantile_bin"))


def gender_rows(patients_df):
    # same KEY as in the vocabulary: source "patients", label female/male, code F/M
    return (patients_df
            .select(
                "subject_id",
                pl.lit("patients").alias("source"),
                pl.lit(None, pl.String).alias("ontology_version"),
                pl.when(pl.col("gender") == "F").then(pl.lit("female"))
                  .when(pl.col("gender") == "M").then(pl.lit("male"))
                  .otherwise(pl.col("gender").str.to_lowercase()).alias("label"),
                pl.col("gender").alias("code"),
                pl.lit(0, pl.Int64).alias("piece"),
                pl.lit("gender").alias("type"),
            )
            .join(concept_ids, on=KEY, how="left", nulls_equal=True)
            .with_columns(token_id=pl.col("concept_id").fill_null(UNK_ID))
            .drop("concept_id"))

## 6. Build the master table for a group of patients

`build_master` runs all the steps for one bucket of patients, or for a list of patients (for testing).

1. read the events of these patients, without undated events
2. seconds in day, token ids, time order, piece, explode into tokens
3. add the gender token, only for patients who have events
4. sort by patient and piece (the order inside a piece is kept), number the tokens (`pos`), add the split

In [ ]:
def build_master(bucket=None, subject_ids=None):
    events = pl.scan_parquet(EVENTS).select(cfg["cols_input_version"])
    pats = patients
    if bucket is not None:
        events, pats = events.filter(bucket_of() == bucket), pats.filter(bucket_of() == bucket)
    if subject_ids is not None:
        events, pats = (events.filter(pl.col("subject_id").is_in(subject_ids)),
                        pats.filter(pl.col("subject_id").is_in(subject_ids)))

    events = (events
              .filter(pl.col("days_since_birth").is_not_null())   # undated events cannot be placed in time
              .rename({"days_since_birth": "age_days"})
              .collect(engine="streaming")
              .pipe(add_seconds_in_day)
              .pipe(add_token_ids)
              .pipe(order_events)
              .pipe(add_piece)
              .pipe(explode_event))

    # gender only for patients with at least one event: the others are dropped
    static = gender_rows(pats.join(events.select("subject_id").unique(), on="subject_id", how="semi"))

    return (pl.concat([static, events], how="diagonal_relaxed")
            .sort(["subject_id", "piece"], maintain_order=True)   # keeps concept, quantile, [VALUE], unit order
            .with_columns(pos=pl.int_range(pl.len(), dtype=pl.Int64).over("subject_id"))
            .join(split, on="subject_id", how="left")
            .sort(["subject_id", "pos"])                          # a join does not guarantee the row order
            .select(cfg["cols_master_version"]))

## 7. Test on one patient

Check before the full run: the first row is the gender (piece 0, no time), the death is there with its
token id, numeric events have 3 or 4 tokens in the right order, and no token is `[UNK]`.

In [ ]:
example = build_master(subject_ids=[10888222])
print(example.group_by("type").len().sort("len", descending=True))
print("tokens with [UNK]:", example.filter(pl.col("token_id") == UNK_ID).height)
print(example.filter(pl.col("label") == "death"))
example.head(20)

## 8. Full run, bucket by bucket

One bucket of patients at a time (about 1/`n_buckets` of the events), so memory stays low.

- **Memory**: the table of a bucket is deleted after it is written, before the next one is built.
  If the kernel still crashes, raise `n_buckets` in `trajectory_columns.yaml`.
- **Resume after a crash**: a bucket whose file already exists is skipped, so just run the cell again.
  Each file is first written as `.tmp`, then renamed: a crash can never leave a half-written file.
- The file names contain the number of buckets (`master_n128_b000.parquet`): changing `n_buckets`
  puts patients in other buckets, so files of another bucket count are never mixed in.

In [ ]:
for b in range(N_BUCKETS):
    path = MASTER / f"master_n{N_BUCKETS}_b{b:03d}.parquet"
    if path.exists():                                  # already done before a crash: skip
        continue
    master = build_master(bucket=b)
    tmp = path.with_suffix(".tmp")
    master.write_parquet(tmp)
    tmp.replace(path)                                  # rename only once the file is complete
    print(f"bucket {b + 1}/{N_BUCKETS}: {master['subject_id'].n_unique():,} patients, "
          f"{master.height:,} tokens", flush=True)
    del master                                         # free the memory before the next bucket
    gc.collect()

### Patient index

One row per patient: split, number of tokens and bucket, read back from the written files (streamed).

In [ ]:
master_all = pl.scan_parquet(MASTER / f"master_n{N_BUCKETS}_b*.parquet")

patient_index = (master_all
                 .group_by("subject_id")
                 .agg(pl.col("split").first(), pl.len().alias("n_tokens"))
                 .with_columns(bucket=bucket_of())
                 .sort("subject_id")
                 .collect(engine="streaming"))
patient_index.write_parquet(MASTER / f"patient_index_n{N_BUCKETS}.parquet")
patient_index.group_by("split").agg(pl.len().alias("patients"), pl.col("n_tokens").sum()).sort("split")

## 9. Checks on the full master table

- tokens per type, and how many are `[UNK]` (expected: 0)
- every patient starts with the gender (pos 0)
- number of death tokens

In [ ]:
print(master_all.group_by("type").agg(pl.len().alias("tokens"),
                                      (pl.col("token_id") == UNK_ID).sum().alias("unk"))
      .sort("tokens", descending=True).collect(engine="streaming"))
print(master_all.filter(pl.col("pos") == 0).group_by("type").len().collect(engine="streaming"))
print("death tokens:", master_all.filter(pl.col("label") == "death").select(pl.len()).collect().item())

## 10. D1, D2 and D3 from the master table

The models differ only by the rows (token types) and the columns they keep, both read from `models`
in `trajectory_columns.yaml`. After the row filter `pos` is counted again; `piece` stays.
`model_id` is the token id after the cut-off of section 2: this is what the model's embedding table reads.

| model | token types kept | extra columns |
|---|---|---|
| D1 | gender, code | – |
| D2 | gender, code, quantile, unit | – |
| D3 | gender, code, quantile, value, unit | value, plus token_id and unit to scale it |

In [ ]:
def model_view(master, model, remap):
    spec = cfg["models"][model]
    return (master
            .filter(pl.col("type").is_in(spec["types"]))
            .with_columns(
                pos=pl.int_range(pl.len(), dtype=pl.Int64).over("subject_id"),
                # full id -> model id (rare concepts -> [UNK]); replace_strict keeps the row order
                model_id=pl.col("token_id").replace_strict(remap["token_id"], remap["model_id"]),
            )
            .select(spec["cols"]))


for model in ["D1", "D2", "D3"]:
    view = model_view(example, model, remap)
    print(f"{model}: {view.height} tokens, columns: {view.columns}")

model_view(example, "D2", remap)